In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
import matplotlib.pyplot as plt
import numpy as np

In [3]:
from pathlib import Path

DATASET_PATH = Path("C:/Users/karthii/OneDrive/Desktop/crop model/datacrop")
img_size = 224
batch_size = 64

In [4]:
train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2
)

In [5]:
train_data = train_gen.flow_from_directory(
    DATASET_PATH,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

Found 7304 images belonging to 8 classes.


In [6]:
val_data = train_gen.flow_from_directory(
    DATASET_PATH,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 1823 images belonging to 8 classes.


In [7]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

In [8]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

In [9]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(8, activation='softmax')
])

In [10]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [12]:
history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10
229/229 ━━━━━━━━━━━━━━━━━━━━ 1130s 5s/step - accuracy: 0.5047 - loss: 1.4441 - val_accuracy: 0.5623 - val_loss: 1.2201
Epoch 2/10
229/229 ━━━━━━━━━━━━━━━━━━━━ 4170s 17s/step - accuracy: 0.7976 - loss: 0.6513 - val_accuracy: 0.7543 - val_loss: 0.7146
Epoch 3/10
229/229 ━━━━━━━━━━━━━━━━━━━━ 467s 2s/step - accuracy: 0.8621 - loss: 0.4315 - val_accuracy: 0.8470 - val_loss: 0.4703
Epoch 4/10
229/229 ━━━━━━━━━━━━━━━━━━━━ 427s 2s/step - accuracy: 0.8880 - loss: 0.3479 - val_accuracy: 0.8815 - val_loss: 0.3424
Epoch 5/10
229/229 ━━━━━━━━━━━━━━━━━━━━ 486s 2s/step - accuracy: 0.9070 - loss: 0.2869 - val_accuracy: 0.8958 - val_loss: 0.2980
Epoch 6/10
229/229 ━━━━━━━━━━━━━━━━━━━━ 564s 2s/step - accuracy: 0.9169 - loss: 0.2465 - val_accuracy: 0.9177 - val_loss: 0.2283
Epoch 7/10
229/229 ━━━━━━━━━━━━━━━━━━━━ 574s 2s/step - accuracy: 0.9274 - loss: 0.2212 - val_accuracy: 0.9320 - val_loss: 0.1932
Epoch 8/10
229/229 ━━━━━━━━━━━━━━━━━━━━ 603s 3s/step - accuracy: 0.9395 - loss: 0.1892 - val_a

In [16]:
model.save("crop_model.keras")

In [15]:
model.save("crop_disease_model2.h5")